In [2]:
import numpy as np
import pandas as pd


In [3]:
df = pd.read_csv("data/sessions_raw.csv")


In [28]:
df.head(10)

,user_pseudo_id,session_id,event_date,device_type,country,traffic_source,traffic_medium,event_count,pageviews,transactions,revenue,add_to_cart_events,session_start,session_end
0,1.131277e+06,6558256025,20201101,mobile,Canada,google,organic,4,1,0,NaN,0,1604258186251809,1604258194408372
1,1.202440e+06,2831108112,20201101,desktop,Canada,(direct),(none),7,2,0,NaN,0,1604208979592943,1604209020830092
2,1.202440e+06,803541518,20201101,desktop,Canada,shop.googlemerchandisestore.com,referral,3,1,0,NaN,0,1604216352814349,1604216362457383
3,1.264630e+06,407300370,20201101,desktop,Canada,(direct),(none),11,2,0,NaN,0,1604250088161864,1604250164382121
4,1.271864e+06,2218414804,20201101,mobile,India,shop.googlemerchandisestore.com,referral,3,1,0,NaN,0,1604252355306063,1604252360742752
5,1.323679e+06,7023276138,20201101,mobile,Hong Kong,(data deleted),(data deleted),4,1,0,NaN,0,1604202416502537,1604202423050321
6,1.358307e+06,3145608691,20201101,mobile,Australia,<Other>,<Other>,4,1,0,NaN,0,1604247741685420,1604247756332764
7,1.370782e+06,1108494361,20201101,desktop,India,<Other>,<Other>,5,1,0,NaN,0,1604257518132840,1604257627443136
8,1.390792e+06,7630153970,20201101,mobile,United States,shop.googlemerchandisestore.com,referral,21,6,0,NaN,0,1604247965040363,1604248044565221
9,1.511634e+06,9101062106,20201101,mobile,United States,google,organic,12,4,0,NaN,0,1604229312748361,1604229423066522


In [19]:
col_names = df.columns.tolist()
print(col_names)

col_nulls = {}

for col in df.columns:
    col_nulls[col] = df[col].isnull().sum() / len(df)

col_nulls_df = pd.DataFrame(list(col_nulls.items()), columns=['column names', 'null_percentage'])
col_nulls_df.head(20)

['user_pseudo_id', 'session_id', 'event_date', 'device_type', 'country', 'traffic_source', 'traffic_medium', 'event_count', 'pageviews', 'transactions', 'revenue', 'add_to_cart_events', 'session_start', 'session_end']


,column names,null_percentage
0,user_pseudo_id,0.000000
1,session_id,0.000000
2,event_date,0.000000
3,device_type,0.000000
4,country,0.000000
5,traffic_source,0.000000
6,traffic_medium,0.000000
7,event_count,0.000000
8,pageviews,0.000000
9,transactions,0.000000


In [ ]:
col_datatypes = df.dtypes
col_datatype

user_pseudo_id        float64
session_id              int64
event_date              int64
device_type               str
country                   str
traffic_source            str
traffic_medium            str
event_count             int64
pageviews               int64
transactions            int64
revenue               float64
add_to_cart_events      int64
session_start           int64
session_end             int64
dtype: object

In [29]:
df = df[df['transactions'] > 0]
df.head()

,user_pseudo_id,session_id,event_date,device_type,country,traffic_source,traffic_medium,event_count,pageviews,transactions,revenue,add_to_cart_events,session_start,session_end
54,4.075022e+06,2785575045,20201101,mobile,United States,(direct),(none),31,9,1,13.0,0,1604236797820893,1604237080234422
274,7.905300e+09,9969096643,20201101,desktop,United States,shop.googlemerchandisestore.com,referral,47,17,1,120.0,0,1604263051264978,1604263711863736
424,3.302728e+07,3130409540,20201101,desktop,United Kingdom,google,organic,103,28,1,63.0,0,1604228090461638,1604229942316229
441,3.663870e+07,6944996097,20201101,mobile,United States,<Other>,<Other>,111,38,1,34.0,0,1604189395117321,1604190463115955
509,6.868924e+07,4529793128,20201101,desktop,United States,shop.googlemerchandisestore.com,referral,162,51,1,59.0,0,1604251485937856,1604252862237560


In [9]:
df["session_duration_sec"] = ((df["session_end"] - df["session_start"]) / 1_000_000).clip(lower=0)

# if duration is neg replace w 0
df["session_duration_sec"] = df["session_duration_sec"].fillna(0)

df["engagement_score"] = (
    np.log1p(df["pageviews"]) * 0.4   # pageviews have highest weight because bigger sign of engagement
    + np.log1p(df["event_count"]) * 0.3
    + np.log1p(df["session_duration_sec"]) * 0.3
)
example = df.groupby('user_pseudo_id')['engagement_score']

example.tail(20)

0         1.424432
1         2.186267
2         1.402619
3         2.488915
4         1.251751
            ...   
360969    1.474966
360970    2.779566
360971    0.693147
360972    5.658198
360973    2.519545
Name: engagement_score, Length: 360974, dtype: float64

In [10]:
df_clean = pd.read_csv("data/sessions_clean.csv")



In [15]:
df_clean.head(10)


,user_pseudo_id,session_id,event_date,device_type,country,traffic_source,traffic_medium,event_count,pageviews,transactions,...,session_end,converted,session_duration_sec,is_bounce_proxy,engagement_score,prior_sessions,prior_pageviews,prior_revenue,prior_transactions,prior_avg_engagement
0,1.000136e+07,5983736405,2020-12-12,desktop,SWEDEN,<other>,organic,4,1,0,...,1607746292809866,0,15.301788,0,1.597473,0,0,0.0,0,0.000000
1,1.000223e+09,6063162078,2021-01-07,mobile,AUSTRALIA,(direct),(none),6,2,0,...,1610044544561794,0,10.472407,0,1.755201,0,0,0.0,0,0.000000
2,1.000300e+06,3338398581,2021-01-20,desktop,INDIA,(direct),(none),5,2,0,...,1611140705779925,0,5.892678,0,1.556111,0,0,0.0,0,0.000000
3,1.000300e+06,9350310735,2020-11-04,desktop,FRANCE,<other>,<other>,4,1,0,...,1604481678760672,0,7.103658,1,1.387785,0,0,0.0,0,0.000000
4,1.000300e+06,3614622791,2020-11-04,desktop,FRANCE,shop.googlemerchandisestore.com,referral,2,0,0,...,1604497410570448,0,0.000000,1,0.329584,1,1,0.0,0,1.387785
5,1.000303e+07,699335171,2020-12-11,desktop,UNITED STATES,(direct),(none),3,1,0,...,1607707512071831,0,0.000000,1,0.693147,0,0,0.0,0,0.000000
6,1.000436e+07,39098263,2021-01-08,mobile,RUSSIA,google,cpc,5,2,0,...,1610079022482922,0,10.110499,0,1.699340,0,0,0.0,0,0.000000
7,1.000442e+06,6266397503,2020-12-07,mobile,ECUADOR,google,organic,14,4,0,...,1607366325977532,0,62.028796,0,2.699268,0,0,0.0,0,0.000000
8,1.000534e+07,1243001234,2020-12-12,desktop,CHINA,google,cpc,18,5,0,...,1607759237122357,0,516.523362,0,3.474752,0,0,0.0,0,0.000000
9,1.000534e+07,3552270955,2020-12-12,desktop,CHINA,(data deleted),(data deleted),2,0,0,...,1607763539088457,0,0.000000,1,0.329584,1,5,0.0,0,3.474752


In [18]:
df_clean.columns.tolist()

['user_pseudo_id',
 'session_id',
 'event_date',
 'device_type',
 'country',
 'traffic_source',
 'traffic_medium',
 'event_count',
 'pageviews',
 'transactions',
 'revenue',
 'add_to_cart_events',
 'session_start',
 'session_end',
 'converted',
 'session_duration_sec',
 'is_bounce_proxy',
 'engagement_score',
 'prior_sessions',
 'prior_pageviews',
 'prior_revenue',
 'prior_transactions',
 'prior_avg_engagement']

In [19]:
df.columns.tolist()

['user_pseudo_id',
 'session_id',
 'event_date',
 'device_type',
 'country',
 'traffic_source',
 'traffic_medium',
 'event_count',
 'pageviews',
 'transactions',
 'revenue',
 'add_to_cart_events',
 'session_start',
 'session_end',
 'session_duration_sec',
 'engagement_score']